# Cons in this data preparation
1. We are removing the URL, which is essential for ref
2. We haven't deep dive into how the html elements are removed

In [2]:
import pandas as pd
import re
from html import unescape
from datetime import datetime, timezone

In [3]:
df_questions = pd.read_csv('data/Questions.csv', encoding='latin-1')
df_answers = pd.read_csv('data/Answers.csv', encoding='latin-1')

In [4]:
df_questions = df_questions[["Id", "CreationDate", "Title", "Body"]].rename({"Id": "QuestionId", "Body": "QuestionBody", "CreationDate": "QuestionCreationDate"}, axis=1)
df_answers = df_answers[["Id", "CreationDate", "ParentId", "Body"]].rename({"Id": "AnswerId", "Body": "AnswerBody", "CreationDate": "AnswerCreationDate"}, axis=1)

In [5]:
df_stack_data = df_questions.merge(df_answers, how='left', left_on='QuestionId', right_on='ParentId')

In [6]:
df_stack_data.drop(["ParentId"], axis=1, inplace=True)

In [7]:
def remove_html_tags(text):
    if pd.isna(text):
        return text
    text = unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_stack_data["AnswerBody"] = df_stack_data["AnswerBody"].apply(remove_html_tags)
df_stack_data["QuestionBody"] = df_stack_data["QuestionBody"].apply(remove_html_tags)

In [8]:
# Limiting data to questions created after 2014-01-01
# Reducing data from 2176163 to 952556
df_stack_data["QuestionCreationDate"] = pd.to_datetime(df_stack_data["QuestionCreationDate"], errors='coerce')
df_stack_data = df_stack_data[df_stack_data["QuestionCreationDate"] >= datetime(2014, 1, 1, tzinfo= timezone.utc)]

In [9]:
df_stack_data.dropna(subset=["AnswerBody"], inplace=True)

In [10]:
df_stack_data = pd.DataFrame(df_stack_data[["QuestionId", "Title", "QuestionBody", "AnswerBody"]].groupby(["QuestionId", "Title", "QuestionBody"])["AnswerBody"].apply(list).reset_index())

In [11]:
df_stack_data.columns

Index(['QuestionId', 'Title', 'QuestionBody', 'AnswerBody'], dtype='str')

In [12]:
def format_stack_data(row):
    answers = " | ".join(row["AnswerBody"]) if isinstance(row["AnswerBody"], list) else ""
    return (
        f"Question title: {row['Title']}\n"
        f"Question body: {row['QuestionBody']}\n"
        f"Answers: {answers}"
    )

formatted_rows = df_stack_data.apply(format_stack_data, axis=1)
formatted_rows.head()

0    Question title: PHP mailing address preg_match...
1    Question title: generating json for google cha...
2    Question title: Polymorphism and inheritance i...
3    Question title: Microsoft Word Highlighting Te...
4    Question title: How do you write a pl sql scri...
dtype: str

In [13]:
formatted_rows.to_csv("data/formatted_stack_data.csv", index=False, header=False, encoding='utf-8')

In [14]:
# df_stack_data.to_csv('data/StackOverflow_Cleaned.csv', index=False, encoding='utf-8')